# Matrix Operations

矩阵运算与线性变换。从向量点积到矩阵乘法，从转置到逆矩阵，最终用 2D 线性变换可视化建立几何直觉。

> 本节课代码为主，每个数学概念都配多个可运行实例。建议逐段运行，观察 shape 和数值变化。

## 0. 环境配置与导入

确认 PyTorch 可用，设置随机种子保证可复现。

In [ ]:
import torch
import matplotlib.pyplot as plt

print("PyTorch version:", torch.__version__)
torch.manual_seed(42)  # 固定随机种子，保证每次运行结果一致

## 1. 向量点积与范数

### 1.1 点积的数学定义

两个等长向量 $a, b \in \mathbb{R}^n$ 的点积（dot product，也称内积 inner product）：

$$
a \cdot b = \langle a, b \rangle = \sum_{i=1}^{n} a_i b_i
$$

几何意义：

$$
a \cdot b = \|a\|_2 \|b\|_2 \cos \theta
$$

其中 $\theta$ 是两向量的夹角。点积为 0 ⟺ 两向量正交（垂直）。

In [ ]:
a = torch.tensor([1.0, 2.0, 3.0])
b = torch.tensor([4.0, 5.0, 6.0])

# torch.dot：只接受两个 1D 张量，长度必须相等
dot_result = torch.dot(a, b)
print("a =", a)
print("b =", b)
print("torch.dot(a, b) =", dot_result)   # 1*4 + 2*5 + 3*6 = 4+10+18 = 32

# 手算验证
manual = sum(ai * bi for ai, bi in zip(a.tolist(), b.tolist()))
print("手算验证 Σ a_i*b_i =", manual)

### 1.2 点积的几何意义：正交验证

点积 $= 0$ 意味着两向量夹角 $\theta = 90°$，即正交。

- 同向向量：点积 > 0
- 反向向量：点积 < 0
- 正交向量：点积 = 0

In [ ]:
# 同向：点积为正
v1 = torch.tensor([1.0, 0.0])
v2 = torch.tensor([2.0, 0.0])
print("同向 dot:", torch.dot(v1, v2))   # 1*2 + 0*0 = 2 > 0

# 反向：点积为负
v3 = torch.tensor([-1.0, 0.0])
print("反向 dot:", torch.dot(v1, v3))   # 1*(-1) + 0*0 = -1 < 0

# 正交：点积为 0
v4 = torch.tensor([0.0, 1.0])
print("正交 dot:", torch.dot(v1, v4))   # 1*0 + 0*1 = 0

# 非正交非平行
v5 = torch.tensor([1.0, 1.0])
print("夹角45° dot:", torch.dot(v1, v5))  # 1*1 + 0*1 = 1 > 0

### 1.3 范数 Norm

向量的范数衡量向量的"长度"或"大小"。常见范数：

| 范数 | 公式 | 含义 |
|------|------|------|
| L1 | $\|x\|_1 = \sum_i \|x_i\|$ | 绝对值之和（曼哈顿距离） |
| L2 | $\|x\|_2 = \sqrt{\sum_i x_i^2}$ | 欧几里得长度 |
| Lp | $\|x\|_p = (\sum_i \|x_i\|^p)^{1/p}$ | 通用形式 |
| L∞ | $\|x\|_\infty = \max_i \|x_i\|$ | 最大绝对值 |

**L2 范数与点积的关系**：$\|x\|_2 = \sqrt{x \cdot x}$

In [ ]:
x = torch.tensor([3.0, -4.0])

# L2 范数（默认）：sqrt(3^2 + (-4)^2) = sqrt(25) = 5
l2 = torch.linalg.norm(x)
print("L2 范数:", l2.item())

# 用点积验证 L2：sqrt(dot(x, x))
l2_by_dot = torch.sqrt(torch.dot(x, x))
print("用点积算 L2:", l2_by_dot.item())

# L1 范数：|3| + |-4| = 7
l1 = torch.linalg.norm(x, ord=1)
print("L1 范数:", l1.item())

# L∞ 范数：max(|3|, |-4|) = 4
linf = torch.linalg.norm(x, ord=float('inf'))
print("L∞ 范数:", linf.item())

# L3 范数：(|3|^3 + |-4|^3)^(1/3) = (27+64)^(1/3) = 91^(1/3) ≈ 4.498
l3 = torch.linalg.norm(x, ord=3)
print("L3 范数:", l3.item())

### 1.4 余弦相似度

余弦相似度衡量两个向量方向的相似程度，与向量长度无关：

$$
\operatorname{cos\_sim}(a, b) = \frac{a \cdot b}{\|a\|_2 \|b\|_2} = \cos \theta
$$

取值范围 $[-1, 1]$：1 = 完全同向，0 = 正交，-1 = 完全反向。

In [ ]:
def cosine_similarity(a, b):
    dot = torch.dot(a, b)
    norm_a = torch.linalg.norm(a)
    norm_b = torch.linalg.norm(b)
    return dot / (norm_a * norm_b)

# 完全同向 → 1
a1 = torch.tensor([1.0, 2.0])
b1 = torch.tensor([2.0, 4.0])  # = 2 * a1
print("完全同向:", cosine_similarity(a1, b1).item())

# 正交 → 0
a2 = torch.tensor([1.0, 0.0])
b2 = torch.tensor([0.0, 1.0])
print("正交:", cosine_similarity(a2, b2).item())

# 完全反向 → -1
a3 = torch.tensor([1.0, 1.0])
b3 = torch.tensor([-1.0, -1.0])
print("完全反向:", cosine_similarity(a3, b3).item())

# 长度不同但方向相同 → 仍为 1（余弦相似度与长度无关）
a4 = torch.tensor([1.0, 0.0])
b4 = torch.tensor([100.0, 0.0])
print("方向相同长度不同:", cosine_similarity(a4, b4).item())

### 1.5 torch.dot vs torch.inner

- `torch.dot(a, b)`：严格要求两个 **1D** 张量，返回标量
- `torch.inner(a, b)`：1D 时与 dot 相同；高维时沿**最后一维**计算内积，支持广播

高维时 `inner` 的行为：对两个张量的最后一维做点积，前面的维度广播。

In [ ]:
# 1D 时完全相同
a = torch.tensor([1.0, 2.0, 3.0])
b = torch.tensor([4.0, 5.0, 6.0])
print("1D dot:  ", torch.dot(a, b))
print("1D inner:", torch.inner(a, b))

# 高维时 inner 沿最后一维计算
A = torch.tensor([[1.0, 2.0], [3.0, 4.0]])   # [2, 2]
B = torch.tensor([[5.0, 6.0], [7.0, 8.0]])   # [2, 2]
inner_result = torch.inner(A, B)
print("\n高维 inner result shape:", inner_result.shape)  # [2, 2]
print(inner_result)
# inner_result[i,j] = A[i,:] · B[j,:]
# [0,0] = 1*5 + 2*6 = 17
# [0,1] = 1*7 + 2*8 = 23
# [1,0] = 3*5 + 4*6 = 39
# [1,1] = 3*7 + 4*8 = 53

# dot 不支持高维（取消注释查看报错）
# torch.dot(A, B)
# RuntimeError: 1D tensors expected

### 1.6 易错点汇总

1. `torch.dot` 只接受 1D，高维用 `torch.inner` 或 `torch.matmul`
2. 两向量长度必须相等，否则报错
3. 整数张量也可以 dot，但结果是整数；需要浮点时先 `.float()`

In [ ]:
# 长度不匹配 → 报错
a = torch.tensor([1.0, 2.0, 3.0])
b = torch.tensor([1.0, 2.0])
try:
    torch.dot(a, b)
except RuntimeError as e:
    print("长度不匹配报错:", e)

# 整数张量 dot
ai = torch.tensor([1, 2, 3])
bi = torch.tensor([4, 5, 6])
print("整数 dot:", torch.dot(ai, bi), "dtype:", torch.dot(ai, bi).dtype)
print("转 float 后 dot:", torch.dot(ai.float(), bi.float()), "dtype:", torch.dot(ai.float(), bi.float()).dtype)

## 2. 矩阵-向量乘法

### 2.1 数学定义

矩阵 $A \in \mathbb{R}^{m \times n}$ 乘以向量 $x \in \mathbb{R}^n$，得到向量 $y \in \mathbb{R}^m$：

$$
y = Ax, \quad y_i = \sum_{j=1}^{n} A_{ij} x_j
$$

**维度要求**：A 的列数 = x 的长度（内维匹配）。

**几何直觉**：矩阵代表一个线性变换，将 $n$ 维空间的向量 $x$ 映射到 $m$ 维空间的向量 $y$。

In [ ]:
A = torch.tensor([[1.0, 2.0],
                  [3.0, 4.0],
                  [5.0, 6.0]])   # [3, 2]
x = torch.tensor([1.0, 1.0])        # [2]

# torch.mv(A, x)：matrix-vector product
y = torch.mv(A, x)
print("A shape:", A.shape, "x shape:", x.shape)
print("y = A @ x =", y, "shape:", y.shape)
# y[0] = 1*1 + 2*1 = 3
# y[1] = 3*1 + 4*1 = 7
# y[2] = 5*1 + 6*1 = 11

# 也可以用 @ 运算符
y2 = A @ x
print("A @ x =", y2)

### 2.2 行视角：每个输出是点积

$y_i = A[i, :] \cdot x$，即输出向量的第 $i$ 个元素 = A 的第 $i$ 行与 $x$ 的点积。

In [ ]:
A = torch.tensor([[1.0, 2.0],
                  [3.0, 4.0],
                  [5.0, 6.0]])
x = torch.tensor([2.0, 3.0])

y = torch.mv(A, x)
print("y =", y)

# 逐行验证：y[i] = dot(A[i], x)
for i in range(A.shape[0]):
    row_dot = torch.dot(A[i], x)
    print(f"y[{i}] = dot(A[{i}], x) = {row_dot.item()}, 匹配: {torch.isclose(y[i], row_dot)}")

### 2.3 列视角：线性组合（最重要的视角）

矩阵-向量乘法可以理解为 **A 的列向量的线性组合**，系数是 $x$ 的元素：

$$
y = x_1 \cdot A[:, 0] + x_2 \cdot A[:, 1] + \cdots + x_n \cdot A[:, n-1]
$$

这个视角是理解线性代数的核心：$Ax$ 的结果落在 A 的列空间（column space）中。

In [ ]:
A = torch.tensor([[1.0, 2.0],
                  [3.0, 4.0],
                  [5.0, 6.0]])
x = torch.tensor([2.0, 3.0])

# 列视角：y = x[0]*A[:,0] + x[1]*A[:,1]
y_by_cols = x[0] * A[:, 0] + x[1] * A[:, 1]
print("列视角计算 y =", y_by_cols)

# 与 mv 结果对比
y_mv = torch.mv(A, x)
print("torch.mv 结果 y =", y_mv)
print("两者相等:", torch.allclose(y_by_cols, y_mv))

# 更一般地，用循环实现列视角
y_manual = torch.zeros(A.shape[0])
for j in range(A.shape[1]):
    y_manual += x[j] * A[:, j]
print("循环列视角 y =", y_manual)

> 平板手写演示：用列视角解释为什么 $Ax$ 只能得到 A 的列向量的线性组合，以及当 A 的列线性相关时，输出空间维度会降低。

### 2.4 维度不匹配与易错点

- A 的列数必须等于 x 的长度
- `torch.mv` 严格要求 A 是 2D、x 是 1D
- 用 `@` 时 PyTorch 会自动处理，但要清楚实际发生了什么

In [ ]:
# 维度不匹配报错
A = torch.tensor([[1.0, 2.0], [3.0, 4.0]])  # [2, 2]
x_wrong = torch.tensor([1.0, 2.0, 3.0])       # [3]，长度不匹配
try:
    torch.mv(A, x_wrong)
except RuntimeError as e:
    print("维度不匹配报错:", e)

# mv 要求 A 必须是 2D
try:
    torch.mv(torch.tensor([1.0, 2.0]), torch.tensor([1.0, 2.0]))
except RuntimeError as e:
    print("A 不是 2D 报错:", e)

# 正确用法
x_correct = torch.tensor([1.0, 2.0])
print("正确结果:", torch.mv(A, x_correct))

## 3. 矩阵-矩阵乘法

### 3.1 数学定义

矩阵 $A \in \mathbb{R}^{m \times n}$ 乘以矩阵 $B \in \mathbb{R}^{n \times p}$，得到 $C \in \mathbb{R}^{m \times p}$：

$$
C = AB, \quad C_{ij} = \sum_{k=1}^{n} A_{ik} B_{kj}
$$

**维度规则**：A 的列数 = B 的行数（内维匹配），输出维度 = (A 的行数, B 的列数)。

记忆口诀：**"中间消掉，留下两边"** —— $[m,n] \times [n,p] \to [m,p]$。

In [ ]:
A = torch.tensor([[1.0, 2.0],
                  [3.0, 4.0]])   # [2, 2]
B = torch.tensor([[5.0, 6.0],
                  [7.0, 8.0]])   # [2, 2]

C = torch.mm(A, B)
print("A shape:", A.shape, "B shape:", B.shape)
print("C = A @ B shape:", C.shape)
print(C)
# C[0,0] = 1*5 + 2*7 = 19
# C[0,1] = 1*6 + 2*8 = 22
# C[1,0] = 3*5 + 4*7 = 43
# C[1,1] = 3*6 + 4*8 = 50

# 非方阵例子
A2 = torch.randn(3, 4)   # [3, 4]
B2 = torch.randn(4, 2)   # [4, 2]
C2 = A2 @ B2
print("\n非方阵: [3,4] @ [4,2] ->", C2.shape)

### 3.2 元素视角：每个 C[i,j] 是点积

$C_{ij} = A[i, :] \cdot B[:, j]$，即 C 的每个元素 = A 的第 i 行与 B 的第 j 列的点积。

In [ ]:
A = torch.tensor([[1.0, 2.0],
                  [3.0, 4.0]])
B = torch.tensor([[5.0, 6.0],
                  [7.0, 8.0]])

C = A @ B
print("C =\n", C)

# 逐元素验证
for i in range(A.shape[0]):
    for j in range(B.shape[1]):
        c_ij = torch.dot(A[i], B[:, j])
        print(f"C[{i},{j}] = dot(A[{i},:], B[:,{j}]) = {c_ij.item()}, 匹配: {torch.isclose(C[i,j], c_ij)}")

### 3.3 行视角：C 的每行 = A 的行 × B

$C[i, :] = A[i, :] @ B$，即 C 的第 i 行 = A 的第 i 行作为行向量乘以 B。

这意味着：**B 决定了如何变换每一行**，A 的每一行经过同一个变换 B 得到 C 的对应行。

In [ ]:
A = torch.tensor([[1.0, 2.0],
                  [3.0, 4.0]])
B = torch.tensor([[5.0, 6.0],
                  [7.0, 8.0]])

C = A @ B
print("C =\n", C)

# 逐行验证
for i in range(A.shape[0]):
    # A[i] 是 1D，需要 unsqueeze 成 [1,2] 才能用 mm
    row_i = A[i].unsqueeze(0) @ B   # [1,2] @ [2,2] -> [1,2]
    print(f"C[{i},:] = {row_i.squeeze(0)}, 匹配: {torch.allclose(C[i], row_i.squeeze(0))}")

### 3.4 列视角：C 的每列 = A × B 的列

$C[:, j] = A @ B[:, j]$，即 C 的第 j 列 = A 乘以 B 的第 j 列。

结合第 2 节的列视角：$C[:, j]$ 是 A 的列向量以 $B[:, j]$ 为系数的线性组合。

**这意味着：C 的每一列都落在 A 的列空间中。**

In [ ]:
A = torch.tensor([[1.0, 2.0],
                  [3.0, 4.0]])
B = torch.tensor([[5.0, 6.0],
                  [7.0, 8.0]])

C = A @ B
print("C =\n", C)

# 逐列验证
for j in range(B.shape[1]):
    col_j = A @ B[:, j]   # [2,2] @ [2] -> [2]
    print(f"C[:,{j}] = {col_j}, 匹配: {torch.allclose(C[:, j], col_j)}")

# 进一步：C[:,0] 是 A 的列的线性组合，系数是 B[:,0]
print("\nC[:,0] 的列视角分解:")
print(f"  B[0,0]*A[:,0] = {B[0,0]} * {A[:,0]} = {B[0,0] * A[:,0]}")
print(f"  B[1,0]*A[:,1] = {B[1,0]} * {A[:,1]} = {B[1,0] * A[:,1]}")
print(f"  求和 = {B[0,0] * A[:,0] + B[1,0] * A[:,1]}")

### 3.5 外积视角（进阶，非常重要）

矩阵乘法还可以分解为**外积之和**：

$$
AB = \sum_{k=1}^{n} A[:, k] \otimes B[k, :] = \sum_{k=1}^{n} \text{outer}(A[:, k], B[k, :])
$$

其中外积 $u \otimes v = u v^T$ 是一个矩阵，每个元素 $(u \otimes v)_{ij} = u_i v_j$。

这个视角揭示了矩阵乘法的本质：**AB 是 n 个秩-1 矩阵的和**。

In [ ]:
A = torch.tensor([[1.0, 2.0],
                  [3.0, 4.0]])
B = torch.tensor([[5.0, 6.0],
                  [7.0, 8.0]])

C = A @ B
print("C = A @ B =\n", C)

# 外积视角：C = Σ_k outer(A[:,k], B[k,:])
C_by_outer = torch.zeros(2, 2)
for k in range(A.shape[1]):
    outer_k = torch.outer(A[:, k], B[k, :])
    print(f"\nouter(A[:,{k}], B[{k},:]) =")
    print(outer_k)
    C_by_outer += outer_k

print("\n外积之和 =\n", C_by_outer)
print("与 A@B 相等:", torch.allclose(C_by_outer, C))

# torch.outer 只接受 1D 向量，结果是 [len(u), len(v)]
u = torch.tensor([1.0, 2.0])
v = torch.tensor([3.0, 4.0, 5.0])
print("\nouter([1,2], [3,4,5]) shape:", torch.outer(u, v).shape)
print(torch.outer(u, v))

> 平板手写演示：外积视角是理解矩阵秩、SVD 分解、低秩近似的基础。AB = n 个秩-1 矩阵之和，当 A 或 B 列/行线性相关时，有效秩 < n。

### 3.6 关键性质

1. **不满足交换律**：$AB \neq BA$（一般情况下）
2. **满足结合律**：$(AB)C = A(BC)$
3. **满足分配律**：$A(B+C) = AB + AC$
4. **与单位矩阵相乘**：$AI = IA = A$
5. **与零矩阵相乘**：$A0 = 0$

In [ ]:
A = torch.tensor([[1.0, 2.0],
                  [3.0, 4.0]])
B = torch.tensor([[5.0, 6.0],
                  [7.0, 8.0]])

# 1. 交换律不成立
print("AB =\n", A @ B)
print("BA =\n", B @ A)
print("AB == BA?", torch.allclose(A @ B, B @ A))

# 2. 结合律成立
C = torch.tensor([[1.0, 0.0], [0.0, 1.0]])
left = (A @ B) @ C
right = A @ (B @ C)
print("\n(AB)C == A(BC)?", torch.allclose(left, right))

# 3. 分配律
D = torch.tensor([[1.0, 1.0], [1.0, 1.0]])
print("A(B+D) == AB+AD?", torch.allclose(A @ (B + D), A @ B + A @ D))

# 4. 单位矩阵
I = torch.eye(2)
print("\nI =\n", I)
print("A @ I == A?", torch.allclose(A @ I, A))
print("I @ A == A?", torch.allclose(I @ A, A))

# 5. 零矩阵
Z = torch.zeros(2, 2)
print("\nA @ zeros == zeros?", torch.allclose(A @ Z, Z))

### 3.7 维度不匹配报错

内维不匹配时，PyTorch 会明确报错。学会读报错信息是必备技能。

In [ ]:
A = torch.randn(3, 4)   # [3, 4]
B = torch.randn(5, 2)   # [5, 2]，内维 4 != 5

try:
    C = A @ B
except RuntimeError as e:
    print("报错信息:", e)
    # 报错会提示：mat1 and mat2 shapes cannot be multiplied (3x4 and 5x2)

# 正确：内维匹配
B_correct = torch.randn(4, 2)
C = A @ B_correct
print("正确结果 shape:", C.shape)

## 4. 乘法函数全对比

PyTorch 提供了多个乘法函数，各有适用场景。下面用一张表和多组实例彻底讲清区别。

| 函数 | 输入要求 | 输出 | 广播支持 | 典型场景 |
|------|---------|------|---------|---------|
| `torch.dot(a, b)` | 两个 1D，等长 | 标量 | 无 | 向量点积 |
| `torch.mv(A, x)` | A 2D，x 1D | 1D | 无 | 矩阵×向量 |
| `torch.mm(A, B)` | 两个 2D | 2D | 无 | 严格矩阵乘法 |
| `torch.bmm(A, B)` | 两个 3D，batch 维相同 | 3D | 无 | 批量矩阵乘法 |
| `torch.matmul` / `@` | 自动分发 | 视输入而定 | batch 维广播 | **通用，推荐** |

### 4.1 torch.dot：1D × 1D → 标量

In [ ]:
a = torch.tensor([1.0, 2.0, 3.0])
b = torch.tensor([4.0, 5.0, 6.0])
print("dot(a, b) =", torch.dot(a, b), "shape:", torch.dot(a, b).shape)

# dot 严格 1D，2D 会报错
A = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
try:
    torch.dot(A, A)
except RuntimeError as e:
    print("2D 用 dot 报错:", e)

### 4.2 torch.mv：2D × 1D → 1D

In [ ]:
A = torch.tensor([[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]])  # [3, 2]
x = torch.tensor([1.0, 2.0])                                      # [2]
y = torch.mv(A, x)
print("mv(A, x) =", y, "shape:", y.shape)  # [3]

# mv 要求 A 必须 2D，x 必须 1D
try:
    torch.mv(x, A)  # 顺序反了
except RuntimeError as e:
    print("顺序错误报错:", e)

### 4.3 torch.mm：2D × 2D → 2D

In [ ]:
A = torch.randn(3, 4)
B = torch.randn(4, 5)
C = torch.mm(A, B)
print("mm(A, B) shape:", C.shape)  # [3, 5]

# mm 不支持广播，严格 2D
A_batch = torch.randn(2, 3, 4)  # 3D
B_batch = torch.randn(2, 4, 5)  # 3D
try:
    torch.mm(A_batch, B_batch)
except RuntimeError as e:
    print("3D 用 mm 报错:", e)

### 4.4 torch.bmm：3D × 3D → 3D（批量矩阵乘法）

`bmm` = batch mm。第一个维度是 batch 维，要求两个张量的 batch 维**完全相同**，不支持广播。

对 batch 中的每个样本独立做矩阵乘法：`C[i] = A[i] @ B[i]`。

In [ ]:
A = torch.randn(2, 3, 4)  # batch=2, [3, 4]
B = torch.randn(2, 4, 5)  # batch=2, [4, 5]
C = torch.bmm(A, B)
print("bmm shape:", C.shape)  # [2, 3, 5]

# 验证：C[0] = A[0] @ B[0]
print("C[0] == A[0] @ B[0]?", torch.allclose(C[0], A[0] @ B[0]))
print("C[1] == A[1] @ B[1]?", torch.allclose(C[1], A[1] @ B[1]))

# batch 维不同会报错
A2 = torch.randn(3, 3, 4)  # batch=3
try:
    torch.bmm(A2, B)  # batch 3 != 2
except RuntimeError as e:
    print("batch 不匹配报错:", e)

### 4.5 torch.matmul / @：通用自动分发

`matmul` 是最灵活的乘法函数，根据输入维度自动分发：

| 输入 | 行为 | 输出 |
|------|------|------|
| 1D × 1D | dot | 标量 |
| 2D × 2D | mm | 2D |
| 1D × 2D | 先在 1D 前 unsqueeze，做完再 squeeze | 1D |
| 2D × 1D | 相当于 mv | 1D |
| N 维 × N 维 | batch 维广播 + 最后两维做 mm | N 维 |

**`@` 运算符就是 `torch.matmul` 的语法糖。**

In [ ]:
# 1D × 1D → dot
a = torch.tensor([1.0, 2.0, 3.0])
b = torch.tensor([4.0, 5.0, 6.0])
print("1D @ 1D =", a @ b, "(标量)")

# 2D × 2D → mm
A = torch.randn(3, 4)
B = torch.randn(4, 5)
print("2D @ 2D shape:", (A @ B).shape)

# 2D × 1D → mv
A2 = torch.randn(3, 4)
x = torch.randn(4)
print("2D @ 1D shape:", (A2 @ x).shape)

# 1D × 2D → 自动 unsqueeze
x2 = torch.randn(4)
B2 = torch.randn(4, 3)
result = x2 @ B2
print("1D @ 2D shape:", result.shape)  # [3]
# 等价于: x2.unsqueeze(0) @ B2 -> [1,3] -> squeeze -> [3]
print("等价验证:", torch.allclose(result, (x2.unsqueeze(0) @ B2).squeeze(0)))

### 4.6 matmul 的 batch 维广播

高维 matmul 中，**最后两维**做矩阵乘法，**前面的所有维度**视为 batch 维并支持广播。

这是 `matmul` 相比 `bmm` 的最大优势：`bmm` 要求 batch 维完全相同，`matmul` 可以广播。

In [ ]:
# batch 维广播示例
A = torch.randn(2, 3, 4)   # batch=[2], 矩阵 [3,4]
B = torch.randn(1, 4, 5)   # batch=[1]，会广播到 2，矩阵 [4,5]
C = A @ B
print("A shape:", A.shape, "B shape:", B.shape)
print("A @ B shape:", C.shape)  # [2, 3, 5]，batch 广播 1→2

# 验证 C[0] 和 C[1] 都用了同一个 B[0]
print("C[0] == A[0] @ B[0]?", torch.allclose(C[0], A[0] @ B[0]))
print("C[1] == A[1] @ B[0]?", torch.allclose(C[1], A[1] @ B[0]))

# 更复杂的广播：A batch [2,1], B batch [1,3] → 输出 batch [2,3]
A2 = torch.randn(2, 1, 3, 4)
B2 = torch.randn(1, 3, 4, 5)
C2 = A2 @ B2
print("\n更复杂广播: A", A2.shape, "@ B", B2.shape, "->", C2.shape)
# C2[i,j] = A2[i,0] @ B2[0,j]

### 4.7 易错点与选择建议

1. **写新代码优先用 `@`（matmul）**，它自动处理所有情况
2. `mm` 不支持广播，需要严格 2D；高维用 `matmul`
3. `bmm` 不支持广播，batch 维必须相同；需要广播时用 `matmul`
4. `dot` 只支持 1D，不要用于高维
5. **逐元素乘法 `*` / `mul` 不是矩阵乘法**！不要混淆

In [ ]:
# 关键区别：逐元素乘法 * vs 矩阵乘法 @
A = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
B = torch.tensor([[5.0, 6.0], [7.0, 8.0]])

print("逐元素乘法 A * B =\n", A * B)
# [1*5, 2*6] = [5, 12]
# [3*7, 4*8] = [21, 32]

print("\n矩阵乘法 A @ B =\n", A @ B)
# [1*5+2*7, 1*6+2*8] = [19, 22]
# [3*5+4*7, 3*6+4*8] = [43, 50]

print("\n两者完全不同！逐元素乘法要求 shape 完全相同，矩阵乘法要求内维匹配。")

## 5. 转置与对称矩阵

### 5.1 三种转置方式

| 方法 | 适用维度 | 说明 |
|------|---------|------|
| `.t()` | 仅 2D | 转置 2D 矩阵，等价于 `.transpose(0, 1)` |
| `.transpose(dim0, dim1)` | 任意 | 交换指定的两个维度 |
| `.permute(dims)` | 任意 | 按指定顺序重新排列所有维度 |

转置不改变数据本身，只改变索引映射（stride），因此是 O(1) 操作。

In [ ]:
A = torch.tensor([[1.0, 2.0, 3.0],
                  [4.0, 5.0, 6.0]])   # [2, 3]

# .t()：仅 2D
print("A shape:", A.shape)
print("A.t() shape:", A.t().shape)  # [3, 2]
print("A.t() =\n", A.t())

# .transpose(dim0, dim1)：交换两个维度
print("\ntranspose(0,1) shape:", A.transpose(0, 1).shape)

# 高维张量用 transpose
B = torch.randn(2, 3, 4)
print("\nB shape:", B.shape)
print("B.transpose(1,2) shape:", B.transpose(1, 2).shape)  # 交换第1、2维 -> [2,4,3]

# .permute：重新排列所有维度
print("B.permute(2,0,1) shape:", B.permute(2, 0, 1).shape)  # [4,2,3]
# permute 必须给出所有维度的新顺序，transpose 只需指定交换的两个

### 5.2 转置的数学性质

1. $(A^T)^T = A$：转置两次回到原矩阵
2. $(A + B)^T = A^T + B^T$：和的转置 = 转置的和
3. $(AB)^T = B^T A^T$：**积的转置 = 转置的逆序乘积**（注意顺序反转！）
4. $(cA)^T = cA^T$：数乘不影响转置

In [ ]:
A = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
B = torch.tensor([[5.0, 6.0], [7.0, 8.0]])

# 性质1: (A^T)^T = A
print("(A.T).T == A?", torch.allclose(A.t().t(), A))

# 性质2: (A+B)^T = A^T + B^T
print("(A+B).T == A.T+B.T?", torch.allclose((A + B).t(), A.t() + B.t()))

# 性质3: (AB)^T = B^T A^T （重点！顺序反转）
left = (A @ B).t()
right = B.t() @ A.t()
print("\n(AB).T =\n", left)
print("B.T @ A.T =\n", right)
print("(AB).T == B.T @ A.T?", torch.allclose(left, right))

# 注意：A.T @ B.T ≠ (AB).T
print("\nA.T @ B.T =\n", A.t() @ B.t())
print("A.T @ B.T == (AB).T?", torch.allclose(A.t() @ B.t(), (A @ B).t()))
# 这是常见错误！顺序不能反

### 5.3 对称矩阵

对称矩阵满足 $A = A^T$，即 $A_{ij} = A_{ji}$，关于主对角线对称。

构造对称矩阵的常用方法：$A_{sym} = \frac{A + A^T}{2}$

In [ ]:
# 任意方阵
A = torch.tensor([[1.0, 2.0],
                  [3.0, 4.0]])

# 构造对称矩阵
A_sym = (A + A.t()) / 2
print("A_sym =\n", A_sym)
print("A_sym == A_sym.T?", torch.allclose(A_sym, A_sym.t()))
# [[1, 2.5], [2.5, 4]] —— 关于对角线对称

# 另一种构造：A @ A.T 一定是对称半正定矩阵
B = torch.randn(3, 4)
C = B @ B.t()   # [3,3]
print("\nB @ B.T 对称?", torch.allclose(C, C.t()))
print("C =\n", C)

# 对角矩阵天然对称
D = torch.diag(torch.tensor([1.0, 2.0, 3.0]))
print("\n对角矩阵对称?", torch.allclose(D, D.t()))
print(D)

### 5.4 重要坑：转置与内存连续性

转置不复制数据，只改变 stride（索引映射），这可能导致张量变得**不连续**（non-contiguous）。

- `.view()` 要求张量在内存中连续，非连续张量调用 `.view()` 会报错
- `.reshape()` 会在需要时自动创建连续副本，更安全
- `.contiguous()` 可以显式创建连续副本

这是第一节（形状操作）没有覆盖的重要细节。

In [ ]:
A = torch.tensor([[1.0, 2.0, 3.0],
                  [4.0, 5.0, 6.0]])   # [2, 3]

A_t = A.t()   # [3, 2]
print("A 是否连续:", A.is_contiguous())       # True
print("A.t() 是否连续:", A_t.is_contiguous())  # False！转置后不连续

# view 在非连续张量上报错
try:
    A_t.view(6)
except RuntimeError as e:
    print("\nview() 报错:", e)

# 解决方案1：contiguous() 后再 view
A_t_cont = A_t.contiguous()
print("\ncontiguous() 后是否连续:", A_t_cont.is_contiguous())  # True
print("contiguous().view(6):", A_t_cont.view(6))

# 解决方案2：直接用 reshape（自动处理连续性）
print("reshape(6):", A_t.reshape(6))

# 关键区别：view 是 O(1) 且不复制（要求连续），reshape 可能复制（不要求连续）
#  contiguous() 会创建数据副本，因此有内存开销

> 平板手写演示：用内存布局解释为什么转置后张量不连续。原矩阵按行存储 [1,2,3,4,5,6]，转置后按列访问的步长不再是 1。

## 6. 迹 · 行列式 · 逆矩阵

### 6.1 迹 Trace

方阵的迹是主对角线元素之和：

$$
\text{tr}(A) = \sum_{i=1}^{n} A_{ii}
$$

性质：
- $\text{tr}(A + B) = \text{tr}(A) + \text{tr}(B)$
- $\text{tr}(cA) = c \cdot \text{tr}(A)$
- $\text{tr}(AB) = \text{tr}(BA)$（即使 $AB \neq BA$，迹也相等！）
- $\text{tr}(A) = \text{tr}(A^T)$

In [ ]:
A = torch.tensor([[1.0, 2.0],
                  [3.0, 4.0]])
B = torch.tensor([[5.0, 6.0],
                  [7.0, 8.0]])

# trace：对角线之和
print("A =\n", A)
print("tr(A) = 1 + 4 =", torch.trace(A).item())

# 性质：tr(AB) = tr(BA)，即使 AB != BA
print("\nAB =\n", A @ B)
print("BA =\n", B @ A)
print("tr(AB) =", torch.trace(A @ B).item())
print("tr(BA) =", torch.trace(B @ A).item())
print("tr(AB) == tr(BA)?", torch.isclose(torch.trace(A @ B), torch.trace(B @ A)))

# tr(A) = tr(A.T)
print("\ntr(A) == tr(A.T)?", torch.isclose(torch.trace(A), torch.trace(A.t())))

# 3x3 例子
C = torch.tensor([[1.0, 2.0, 3.0],
                  [4.0, 5.0, 6.0],
                  [7.0, 8.0, 9.0]])
print("\ntr(C) = 1+5+9 =", torch.trace(C).item())

### 6.2 行列式 Determinant

行列式是方阵的一个标量值，几何意义是**线性变换的体积缩放因子**。

- 2×2：$\det\begin{pmatrix} a & b \\ c & d \end{pmatrix} = ad - bc$
- $\det(A) = 0$ ⟺ A 是奇异矩阵（不可逆，列线性相关，变换降维）
- $\det(AB) = \det(A) \cdot \det(B)$
- $\det(A^T) = \det(A)$
- $\det(A^{-1}) = 1/\det(A)$

In [ ]:
# 2x2 手算验证
A = torch.tensor([[1.0, 2.0],
                  [3.0, 4.0]])
det_A = torch.det(A)
print("A =\n", A)
print("det(A) = 1*4 - 2*3 = 4-6 =", det_A.item())  # -2

# 用 torch.linalg.det（推荐，新 API）
print("torch.linalg.det(A):", torch.linalg.det(A).item())

# 奇异矩阵：det = 0
S = torch.tensor([[1.0, 2.0],
                  [2.0, 4.0]])  # 第二行 = 2 * 第一行，线性相关
print("\n奇异矩阵 S =\n", S)
print("det(S) =", torch.det(S).item(), "（≈0，奇异）")

# 性质：det(AB) = det(A) * det(B)
B = torch.tensor([[5.0, 6.0], [7.0, 8.0]])
print("\ndet(AB) =", torch.det(A @ B).item())
print("det(A)*det(B) =", (torch.det(A) * torch.det(B)).item())
print("相等?", torch.isclose(torch.det(A @ B), torch.det(A) * torch.det(B)))

# det(A.T) = det(A)
print("det(A.T) =", torch.det(A.t()).item(), "== det(A)?", torch.isclose(torch.det(A.t()), torch.det(A)))

# 3x3 行列式
C = torch.tensor([[1.0, 2.0, 3.0],
                  [0.0, 1.0, 4.0],
                  [5.0, 6.0, 0.0]])
print("\n3x3 det(C) =", torch.det(C).item())

### 6.3 逆矩阵 Inverse

方阵 A 的逆矩阵 $A^{-1}$ 满足：

$$
A A^{-1} = A^{-1} A = I
$$

- 只有**非奇异方阵**（det ≠ 0）才有逆矩阵
- $(AB)^{-1} = B^{-1} A^{-1}$（逆序，与转置类似）
- $(A^T)^{-1} = (A^{-1})^T$
- $(A^{-1})^{-1} = A$

In [ ]:
A = torch.tensor([[1.0, 2.0],
                  [3.0, 4.0]])

# 逆矩阵
A_inv = torch.linalg.inv(A)
print("A =\n", A)
print("A_inv =\n", A_inv)

# 验证：A @ A_inv ≈ I
product = A @ A_inv
print("\nA @ A_inv =\n", product)
print("≈ I?", torch.allclose(product, torch.eye(2), atol=1e-6))

# 验证：A_inv @ A ≈ I
print("\nA_inv @ A ≈ I?", torch.allclose(A_inv @ A, torch.eye(2), atol=1e-6))

# 性质：(AB)^-1 = B^-1 @ A^-1
B = torch.tensor([[5.0, 6.0], [7.0, 8.0]])
AB_inv = torch.linalg.inv(A @ B)
B_inv_A_inv = torch.linalg.inv(B) @ torch.linalg.inv(A)
print("\n(AB)^-1 == B^-1 @ A^-1?", torch.allclose(AB_inv, B_inv_A_inv, atol=1e-5))

# 性质：(A^T)^-1 = (A^-1)^T
print("(A.T)^-1 == (A^-1).T?", torch.allclose(torch.linalg.inv(A.t()), A_inv.t(), atol=1e-6))

### 6.4 奇异矩阵求逆报错

奇异矩阵（det = 0）没有逆矩阵，调用 `linalg.inv` 会报错。

In [ ]:
S = torch.tensor([[1.0, 2.0],
                  [2.0, 4.0]])  # det = 0
print("det(S) =", torch.det(S).item())

try:
    torch.linalg.inv(S)
except RuntimeError as e:
    print("奇异矩阵求逆报错:", e)

# 非方阵也没有逆矩阵
R = torch.randn(3, 4)
try:
    torch.linalg.inv(R)
except RuntimeError as e:
    print("非方阵求逆报错:", e)

### 6.5 解线性方程组：linalg.solve

给定 $Ax = b$，求 $x$。

- 方法1：$x = A^{-1} b$（概念清晰，但数值不稳定，不推荐）
- 方法2：`torch.linalg.solve(A, b)`（内部用 LU 分解，数值稳定，推荐）

**实际工程中永远用 `solve`，不要手动算逆矩阵。**

In [ ]:
# 解 3 元一次方程组：
#  x + 2y + 3z = 14
# 2x +  y +  z =  7
# 3x + 4y + 2z = 17
# 预期解：x=1, y=2, z=3

A = torch.tensor([[1.0, 2.0, 3.0],
                  [2.0, 1.0, 1.0],
                  [3.0, 4.0, 2.0]])
b = torch.tensor([14.0, 7.0, 17.0])

# 推荐方法：linalg.solve
x = torch.linalg.solve(A, b)
print("解 x =", x)  # 应接近 [1, 2, 3]

# 验证：A @ x ≈ b
print("验证 A @ x =", A @ x, "≈ b?", torch.allclose(A @ x, b))

# 不推荐方法：inverse @ b
x_by_inv = torch.linalg.inv(A) @ b
print("\n用逆矩阵求解 x =", x_by_inv)
print("两种方法结果接近?", torch.allclose(x, x_by_inv, atol=1e-5))

# 为什么 solve 更好？当 A 接近奇异时，inverse 方法误差更大
# 这里用一个条件数较大的矩阵演示
A_ill = torch.tensor([[1.0, 1.0], [1.0, 1.0001]])  # 接近奇异
b_ill = torch.tensor([2.0, 2.0001])
x_solve = torch.linalg.solve(A_ill, b_ill)
x_inv = torch.linalg.inv(A_ill) @ b_ill
print("\n病态矩阵:")
print("  solve 结果:", x_solve)
print("  inverse 结果:", x_inv)
print("  两者差异:", (x_solve - x_inv).abs().max().item())
# 两者可能有微小差异，solve 更稳定

## 7. 综合应用：2D 线性变换可视化

这一节把前面所有抽象概念落到几何直觉上。2D 平面上的线性变换都可以用一个 2×2 矩阵表示，矩阵的列就是变换后基向量的位置。

常见变换矩阵：

| 变换 | 矩阵 | 说明 |
|------|------|------|
| 缩放 | $\begin{pmatrix} s_x & 0 \\ 0 & s_y \end{pmatrix}$ | x 方向缩放 $s_x$，y 方向缩放 $s_y$ |
| 旋转 | $\begin{pmatrix} \cos\theta & -\sin\theta \\ \sin\theta & \cos\theta \end{pmatrix}$ | 逆时针旋转 $\theta$ 弧度 |
| 剪切 | $\begin{pmatrix} 1 & k \\ 0 & 1 \end{pmatrix}$ | 水平剪切，x 方向偏移量与 y 成正比 |
| 反射 | $\begin{pmatrix} 1 & 0 \\ 0 & -1 \end{pmatrix}$ | 关于 x 轴反射 |

**复合变换 = 矩阵乘法**，注意顺序：先应用 A 再应用 B = $B \cdot A$（右乘先执行）。

In [ ]:
%matplotlib inline
import torch
import matplotlib.pyplot as plt
import numpy as np

def plot_transform(points, matrix, title, ax=None):
    """绘制变换前后的点集。points: [N, 2] tensor, matrix: [2, 2]"""
    if ax is None:
        fig, ax = plt.subplots(figsize=(5, 5))
    transformed = (matrix @ points.T).T  # [N,2]
    ax.plot(points[:, 0], points[:, 1], 'b-', alpha=0.6, label='原始')
    ax.plot(transformed[:, 0], transformed[:, 1], 'r-', alpha=0.8, label='变换后')
    ax.fill(points[:, 0], points[:, 1], 'b', alpha=0.1)
    ax.fill(transformed[:, 0], transformed[:, 1], 'r', alpha=0.1)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)
    ax.legend()
    ax.set_title(title)
    ax.set_xlim(-4, 4)
    ax.set_ylim(-4, 4)
    return transformed

# 生成一个正方形的顶点（闭合）
square = torch.tensor([[0.0, 0.0], [1.0, 0.0], [1.0, 1.0], [0.0, 1.0], [0.0, 0.0]])
print("正方形点集 shape:", square.shape)
print("正方形:\n", square)

### 7.1 缩放变换

In [ ]:
# 缩放：x 方向 2 倍，y 方向 0.5 倍
S = torch.tensor([[2.0, 0.0],
                  [0.0, 0.5]])
print("缩放矩阵 S =\n", S)
print("det(S) = 面积缩放因子 =", torch.det(S).item())  # 2*0.5 = 1，面积不变

fig, ax = plt.subplots(figsize=(5, 5))
transformed = plot_transform(square, S, "缩放: x→2x, y→0.5y", ax)
plt.tight_layout()
plt.show()

print("\n变换后坐标:\n", transformed)
# 列视角验证：S[:,0]=[2,0] 是变换后的 e1（x轴基向量），S[:,1]=[0,0.5] 是变换后的 e2

### 7.2 旋转变换

In [ ]:
import math

# 旋转：逆时针 45°
theta = math.pi / 4  # 45°
R = torch.tensor([[math.cos(theta), -math.sin(theta)],
                  [math.sin(theta),  math.cos(theta)]])
print("旋转矩阵 R (45°) =\n", R)
print("det(R) =", torch.det(R).item(), "（旋转不改变面积，det=1）")
print("R @ R.T ≈ I?", torch.allclose(R @ R.t(), torch.eye(2), atol=1e-6), "（旋转矩阵是正交矩阵）")

fig, ax = plt.subplots(figsize=(5, 5))
plot_transform(square, R, "旋转: 逆时针 45°", ax)
plt.tight_layout()
plt.show()

# 旋转 90°
theta90 = math.pi / 2
R90 = torch.tensor([[math.cos(theta90), -math.sin(theta90)],
                    [math.sin(theta90),  math.cos(theta90)]])
print("\n旋转 90° 矩阵 =\n", R90)
# [[0, -1], [1, 0]] —— (x,y) → (-y,x)

### 7.3 剪切变换

In [ ]:
# 水平剪切：x' = x + k*y, y' = y
k = 1.0
H = torch.tensor([[1.0, k],
                  [0.0, 1.0]])
print("剪切矩阵 H =\n", H)
print("det(H) =", torch.det(H).item(), "（剪切不改变面积，det=1）")

fig, ax = plt.subplots(figsize=(5, 5))
plot_transform(square, H, f"水平剪切: k={k}", ax)
plt.tight_layout()
plt.show()

# 验证：(0,0)→(0,0), (1,0)→(1,0), (1,1)→(2,1), (0,1)→(1,1)
# 正方形变成平行四边形，面积不变

### 7.4 反射变换

In [ ]:
# 关于 x 轴反射：(x,y) → (x,-y)
F = torch.tensor([[1.0, 0.0],
                  [0.0, -1.0]])
print("反射矩阵 F =\n", F)
print("det(F) =", torch.det(F).item(), "（反射翻转方向，det=-1）")

fig, ax = plt.subplots(figsize=(5, 5))
plot_transform(square, F, "反射: 关于 x 轴", ax)
plt.tight_layout()
plt.show()

# 关于 y=x 反射（转置）：(x,y) → (y,x)
F2 = torch.tensor([[0.0, 1.0],
                   [1.0, 0.0]])
fig, ax = plt.subplots(figsize=(5, 5))
plot_transform(square, F2, "反射: 关于 y=x", ax)
plt.tight_layout()
plt.show()

### 7.5 复合变换：顺序的重要性

复合变换 = 矩阵乘法，但**顺序至关重要**：

- 先旋转再缩放 = $S \cdot R$（R 先作用，S 后作用）
- 先缩放再旋转 = $R \cdot S$（S 先作用，R 后作用）
- 一般 $SR \neq RS$

矩阵在右边的先执行（因为向量是列向量，右乘先作用）。

In [ ]:
import math

# 生成一个长方形（非正方形，更容易看出顺序差异）
rect = torch.tensor([[0.0, 0.0], [2.0, 0.0], [2.0, 1.0], [0.0, 1.0], [0.0, 0.0]])

# 缩放 1.5 倍
S = torch.tensor([[1.5, 0.0], [0.0, 1.5]])
# 旋转 45°
theta = math.pi / 4
R = torch.tensor([[math.cos(theta), -math.sin(theta)],
                  [math.sin(theta),  math.cos(theta)]])

# 先旋转再缩放: S @ R（向量先乘 R，再乘 S）
SR = S @ R
# 先缩放再旋转: R @ S
RS = R @ S

print("S @ R（先旋转再缩放）=\n", SR)
print("R @ S（先缩放再旋转）=\n", RS)
print("SR == RS?", torch.allclose(SR, RS))

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
plot_transform(rect, SR, "先旋转再缩放: S @ R", axes[0])
plot_transform(rect, RS, "先缩放再旋转: R @ S", axes[1])
plt.tight_layout()
plt.show()

# 虽然这里 S 是均匀缩放，SR 和 RS 恰好相等！
# 用非均匀缩放才能看出差异
S2 = torch.tensor([[2.0, 0.0], [0.0, 0.5]])  # x 放大，y 缩小
SR2 = S2 @ R
RS2 = R @ S2
print("\n非均匀缩放:")
print("S2 @ R != R @ S2?", not torch.allclose(SR2, RS2))

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
plot_transform(rect, SR2, "先旋转再非均匀缩放: S2 @ R", axes[0])
plot_transform(rect, RS2, "先非均匀缩放再旋转: R @ S2", axes[1])
plt.tight_layout()
plt.show()

### 7.6 用圆验证变换的几何效果

圆经过线性变换后变成椭圆，椭圆的轴方向和长度由矩阵的特征值和特征向量决定（这是后续课程的内容）。

In [ ]:
# 生成单位圆上的点
t = torch.linspace(0, 2 * math.pi, 100)
circle = torch.stack([torch.cos(t), torch.sin(t)], dim=1)  # [100, 2]

# 任意变换矩阵
M = torch.tensor([[2.0, 1.0],
                  [0.5, 1.5]])

fig, axes = plt.subplots(1, 2, figsize=(11, 5))

# 左图：圆 → 椭圆
transformed = (M @ circle.T).T
axes[0].plot(circle[:, 0], circle[:, 1], 'b-', alpha=0.6, label='单位圆')
axes[0].plot(transformed[:, 0], transformed[:, 1], 'r-', label='变换后(椭圆)')
axes[0].fill(transformed[:, 0], transformed[:, 1], 'r', alpha=0.1)
# 画出变换后的基向量（矩阵的列）
axes[0].arrow(0, 0, M[0, 0], M[1, 0], head_width=0.15, color='green', label='M 的第1列')
axes[0].arrow(0, 0, M[0, 1], M[1, 1], head_width=0.15, color='purple', label='M 的第2列')
axes[0].set_aspect('equal')
axes[0].grid(True, alpha=0.3)
axes[0].legend()
axes[0].set_title("圆 → 椭圆，矩阵列 = 变换后的基向量")
axes[0].set_xlim(-3, 3)
axes[0].set_ylim(-3, 3)

# 右图：det = 面积缩放因子
original_area = math.pi  # 单位圆面积
transformed_area = original_area * abs(torch.det(M).item())
axes[1].text(0.5, 0.5, f"det(M) = {torch.det(M).item():.2f}\n"
                          f"原面积 = π ≈ {original_area:.2f}\n"
                          f"变换后面积 = π×|det| ≈ {transformed_area:.2f}\n"
                          f"面积缩放因子 = |det(M)|",
             fontsize=12, ha='center', va='center', transform=axes[1].transAxes,
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
axes[1].set_title("行列式的几何意义")
axes[1].axis('off')

plt.tight_layout()
plt.show()

print(f"det(M) = {torch.det(M).item():.4f}")
print(f"面积缩放因子 = |det(M)| = {abs(torch.det(M).item()):.4f}")

## 课后练习

### 基础题

1. 设 $a = [1, 2, 3]$，$b = [4, 5, 6]$，用 `torch.dot` 计算点积，并用公式 $a \cdot b = \|a\|\|b\|\cos\theta$ 反求夹角 $\theta$（用 `torch.acos`）。

2. 创建一个 $[4, 3]$ 的随机矩阵 A 和 $[3]$ 的随机向量 x，分别用 `torch.mv` 和列视角（线性组合）计算 $Ax$，验证结果一致。

3. 设 $A = \begin{pmatrix} 1 & 2 \\ 3 & 4 \end{pmatrix}$，$B = \begin{pmatrix} 5 & 6 \\ 7 & 8 \end{pmatrix}$，用元素视角、行视角、列视角、外积视角四种方法分别计算 $AB$，验证四种方法结果相同。

4. 验证矩阵乘法不满足交换律：找两个 2×2 矩阵 A、B，使得 $AB \neq BA$，并计算 $\text{tr}(AB)$ 和 $\text{tr}(BA)$，观察迹是否相等。

### 对比题

5. 分别用 `torch.dot`、`torch.mv`、`torch.mm`、`torch.matmul` 完成以下运算，记录每个函数的输入维度要求和输出 shape：
   - 两个 1D 向量的点积
   - 2D 矩阵 × 1D 向量
   - 两个 2D 矩阵相乘
   - 两个 3D batch 矩阵相乘（batch 维相同）
   - 两个 3D batch 矩阵相乘（batch 维不同，需要广播）

6. 创建一个 $[3, 4]$ 的张量，分别用 `.t()`、`.transpose(0,1)`、`.permute(1,0)` 转置，观察结果是否相同。再创建一个 $[2, 3, 4]$ 的张量，尝试用 `.t()`（会报错），然后用 `.transpose` 和 `.permute` 完成不同的维度交换。

### 几何题

7. 给定变换矩阵 $M = \begin{pmatrix} 1 & 2 \\ 0 & 1 \end{pmatrix}$，预测它对正方形 $[0,1]\times[0,1]$ 的变换效果（是什么变换？面积是否改变？），然后用代码可视化验证。

8. 用矩阵乘法实现"先旋转 30°，再缩放 1.5 倍，再关于 x 轴反射"的复合变换，写出复合矩阵（注意顺序），并用可视化验证。

### 综合题

9. 解以下线性方程组（用 `torch.linalg.solve`），并验证解的正确性：
   $$
   \begin{cases}
   2x + y - z = 1 \\
   x - y + 2z = 5 \\
   3x + 2y + z = 4
   \end{cases}
   $$

10. 数据标准化的矩阵形式：创建一个 $[100, 5]$ 的标准正态随机张量 X（模拟 100 个样本，5 个特征），用矩阵运算实现 $X_{norm} = (X - \mu) \Sigma^{-1}$，其中 $\mu$ 是每列均值（$[1, 5]$），$\Sigma$ 是每列标准差构成的对角矩阵（$[5, 5]$）。验证 $X_{norm}$ 每列均值≈0、标准差≈1。

### autograd 衔接题

11. 设置 `A = torch.randn(3, 3, requires_grad=True)`，`x = torch.randn(3)`，计算 $y = \sum (A @ x)^2$，调用 `backward()`，验证 $\frac{\partial y}{\partial A} = 2 (A @ x) x^T$（用代码手动计算右侧并与 `A.grad` 对比）。

12. 对一个 2×2 矩阵 A 设置 `requires_grad=True`，计算 $loss = \text{det}(A)$，调用 `backward()`，观察 `A.grad`。尝试推导行列式对矩阵元素的梯度公式（伴随矩阵）。